# Clase 2: tablas dinámicas, visualización e interpretación de datos

**Duración:** 180 minutos (3 horas reloj)  
**Modalidad:** explicación, demostración, práctica guiada y desafío integrador  
**Herramientas:** Python, Pandas y Matplotlib  
**Dataset:** ventas de una librería escolar, incluido en el notebook

## Objetivos de aprendizaje

Al finalizar la clase, el estudiante será capaz de:

- construir tablas dinámicas mediante `pivot_table()`;
- elaborar tablas de frecuencias con `pd.crosstab()`;
- seleccionar gráficos adecuados para diferentes preguntas;
- crear y personalizar gráficos con Pandas y Matplotlib;
- interpretar tablas y gráficos sin confundir observaciones con opiniones;
- comunicar hallazgos mediante un miniinforme basado en datos.



## 1. Inicio: una decisión basada en datos 
La librería escolar ya corrigió errores y resumió sus ventas. Ahora la administración debe decidir:

> ¿Qué productos debería promocionar en cada sucursal y cómo podemos explicar la decisión de manera clara?

Una lista extensa de números puede ser correcta, pero difícil de interpretar. Las **tablas dinámicas** reorganizan los datos y los **gráficos** hacen visibles las comparaciones, proporciones y tendencias.

### Preguntas para conversar

1. ¿Es lo mismo registrar muchas operaciones que obtener mayor facturación?
2. ¿Qué resulta más fácil de comprender: 100 filas o un gráfico resumido?
3. ¿Un gráfico bonito siempre representa correctamente la información?

### Ruta del análisis

En la clase anterior llegamos hasta la agrupación. Hoy completaremos la comunicación de resultados:

1. Datos originales.
2. Limpieza.
3. Análisis exploratorio.
4. Agrupación y resumen.
5. Tablas comparativas.
6. Visualización.
7. Interpretación y decisiones.

### Explicación del código: importación de bibliotecas
- `numpy` (`np`) permite generar y trabajar con datos numéricos.
- `pandas` (`pd`) organiza los datos en tablas llamadas DataFrame.
- `matplotlib.pyplot` (`plt`) permite crear y mostrar gráficos.
- `FuncFormatter` personaliza la forma en que aparecen los números en los ejes.
- `pd.set_option()` configura la visualización de las tablas; no modifica los datos.
- `plt.style.use()` aplica un estilo visual uniforme a los gráficos.

In [1]:
# Importación de las herramientas necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
plt.style.use('seaborn-v0_8-whitegrid')

### Dataset de continuidad

La siguiente celda genera y deja limpio el dataset utilizado en la clase anterior. Así todos podrán trabajar sin depender de archivos externos. Cada fila representa una operación de venta.

### Explicación del código: creación del conjunto de datos
- `default_rng(2026)` crea un generador aleatorio con una semilla fija. Así, los resultados se pueden reproducir.
- `n = 120` establece que se generarán 120 registros de venta.
- `productos` contiene los nombres posibles de los artículos.
- `categorias` relaciona cada producto con su categoría y `precios` guarda su precio unitario.
- `rng.choice()` selecciona productos y sucursales; `p` define la probabilidad de selección de cada producto.
- `pd.DataFrame()` construye la tabla con fecha, producto, categoría, cantidad, precio, sucursal y vendedor.
- `pd.to_datetime()` y `pd.to_timedelta()` generan fechas válidas.
- `total_venta` multiplica cantidad por precio unitario.
- `mes` extrae el número del mes y `nombre_mes` lo convierte a texto.
- `head()` muestra las primeras cinco filas para revisar la estructura.

In [2]:
rng = np.random.default_rng(2026)
n = 120
productos = np.array(['Cuaderno', 'Bolígrafo', 'Carpeta', 'Regla', 'Mochila', 'Calculadora'])
categorias = {
    'Cuaderno': 'Papelería', 'Bolígrafo': 'Papelería',
    'Carpeta': 'Papelería', 'Regla': 'Útiles',
    'Mochila': 'Accesorios', 'Calculadora': 'Tecnología'
}
precios = {
    'Cuaderno': 18000, 'Bolígrafo': 5000, 'Carpeta': 12000,
    'Regla': 7000, 'Mochila': 95000, 'Calculadora': 65000
}
producto = rng.choice(productos, size=n, p=[0.25, 0.23, 0.16, 0.12, 0.10, 0.14])
df_limpio = pd.DataFrame({
    'fecha': pd.to_datetime('2026-03-01') + pd.to_timedelta(rng.integers(0, 92, size=n), unit='D'),
    'producto': producto,
    'categoria': [categorias[p] for p in producto],
    'cantidad': rng.integers(1, 8, size=n),
    'precio_unitario': [precios[p] for p in producto],
    'sucursal': rng.choice(['Centro', 'Terminal', 'Mercado'], size=n),
    'vendedor': rng.choice(['Ana', 'Carlos', 'Lucía', 'Miguel'], size=n)
})
df_limpio['total_venta'] = df_limpio['cantidad'] * df_limpio['precio_unitario']
df_limpio['mes'] = df_limpio['fecha'].dt.month
df_limpio['nombre_mes'] = df_limpio['mes'].map({3: 'Marzo', 4: 'Abril', 5: 'Mayo'})
df_limpio.head()

,fecha,producto,categoria,cantidad,precio_unitario,sucursal,vendedor,total_venta,mes,nombre_mes
0,2026-05-30,Cuaderno,Papelería,4,18000,Terminal,Miguel,72000,5,Mayo
1,2026-04-30,Carpeta,Papelería,6,12000,Mercado,Lucía,72000,4,Abril
2,2026-05-31,Bolígrafo,Papelería,2,5000,Centro,Carlos,10000,5,Mayo
3,2026-05-21,Bolígrafo,Papelería,6,5000,Mercado,Carlos,30000,5,Mayo
4,2026-05-08,Bolígrafo,Papelería,2,5000,Mercado,Miguel,10000,5,Mayo


### Activación de conocimientos previos

Antes de avanzar, responda usando Pandas:

1. ¿Cuántas filas y columnas tiene el dataset?
2. ¿Cuál es la facturación total?
3. ¿Qué sucursal tiene mayor facturación?
4. ¿Qué diferencia existe entre `groupby()` y un filtro?

### Explicación del código: dimensiones del dataset
- `df_limpio.shape` devuelve una tupla con dos valores: cantidad de filas y cantidad de columnas.
- La asignación `filas, columnas = ...` guarda cada valor en una variable.
- `print()` muestra los resultados en pantalla.
- En este conjunto de datos se esperan 120 filas y 10 columnas.

In [5]:
# Escriba aquí sus respuestas de recuperación
# ¿Cuántas filas y columnas tiene el dataset?

filas, columnas = df_limpio.shape

print("Filas:", filas)
print("Columnas:", columnas)

Filas: 120
Columnas: 10


### Explicación del código: facturación total
- `df_limpio["total_venta"]` selecciona la columna que contiene el importe de cada operación.
- `.sum()` suma todos los importes.
- El resultado representa la facturación total de las 120 operaciones, no la cantidad de artículos.

In [6]:
#2. ¿Cuál es la facturación total?
facturacion_total = df_limpio["total_venta"].sum()

print("Facturación total:", facturacion_total)

Facturación total: 11396000


### Explicación del código: sucursal con mayor facturación
- `groupby("sucursal")` separa los registros según la sucursal.
- `["total_venta"].sum()` suma la facturación de cada grupo.
- `.idxmax()` devuelve el nombre de la sucursal cuyo total es el mayor.
- Se imprime el nombre; la serie `facturacion_sucursal` también permite revisar los importes de todas las sucursales.

In [10]:
#3. ¿Qué sucursal tiene mayor facturación?
facturacion_sucursal = df_limpio.groupby("sucursal")["total_venta"].sum()

sucursal_mayor = facturacion_sucursal.idxmax()

print("Sucursal con mayor facturación:", sucursal_mayor)

Sucursal con mayor facturación: Centro


4. ¿Qué diferencia existe entre `groupby()` y un filtro?

La diferencia es que groupby agrupa datos por columnas y luego permite realizar operaciones sobre esas columnas, mientras que un filtro es para seleccionar las filas que cumplen x condicion


## 2. Tablas dinámicas con `pivot_table()` 

### Concepto principal

Una **tabla dinámica** resume una gran cantidad de registros y permite compararlos mediante filas y columnas.

Su estructura básica es:

```python
pd.pivot_table(
    datos,
    values='columna_numerica',
    index='filas',
    columns='columnas',
    aggfunc='operacion',
    fill_value=0
)
```

| Parámetro | Función |
|---|---|
| `values` | Indica qué valores numéricos se resumirán |
| `index` | Define las categorías que aparecerán en las filas |
| `columns` | Define las categorías que aparecerán en las columnas |
| `aggfunc` | Establece la operación: suma, promedio, conteo, etc. |
| `fill_value` | Reemplaza combinaciones vacías, normalmente por cero |

### Ejemplo 1: facturación por sucursal y categoría

Pregunta: **¿Cuánto facturó cada categoría en cada sucursal?**

### Explicación del código: tabla dinámica por sucursal y categoría
- `pd.pivot_table()` construye una tabla resumen.
- `df_limpio` es la fuente de datos.
- `values="total_venta"` indica qué columna se va a resumir.
- `index="sucursal"` coloca las sucursales en las filas.
- `columns="categoria"` coloca las categorías en las columnas.
- `aggfunc="sum"` suma la facturación de cada combinación.
- `fill_value=0` muestra cero cuando no existen registros para una combinación.

In [11]:
tabla_sucursal_categoria = pd.pivot_table(
    df_limpio,
    values='total_venta',
    index='sucursal',
    columns='categoria',
    aggfunc='sum',
    fill_value=0
)
tabla_sucursal_categoria

categoria,Accesorios,Papelería,Tecnología,Útiles
sucursal,,,,
Centro,1710000,1367000,845000,196000
Mercado,1045000,1527000,1170000,98000
Terminal,1045000,844000,1430000,119000


### Cómo leer la tabla

- Una **fila** contiene los resultados de una sucursal.
- Una **columna** contiene los resultados de una categoría.
- Cada **celda** es la suma de `total_venta` de esa combinación.
- El cero indica que no hubo registros para la combinación.

No basta con mostrar la tabla: debemos convertir sus números en afirmaciones verificables.

### Ejemplo 2: unidades vendidas por producto y mes


### Explicación del código: unidades por producto y mes
- `values="cantidad"` indica que se sumarán unidades, no dinero.
- `index="producto"` organiza los productos en filas.
- `columns="nombre_mes"` crea una columna por mes.
- `aggfunc="sum"` suma las unidades vendidas.
- `reindex(columns=[...])` fija el orden cronológico de los meses. Sin esta indicación, Pandas podría mostrarlos en orden alfabético.

In [ ]:
tabla_producto_mes = pd.pivot_table(
    df_limpio, values='cantidad', index='producto', columns='nombre_mes',
    aggfunc='sum', fill_value=0
).reindex(columns=['Marzo', 'Abril', 'Mayo'])
tabla_producto_mes

### Ejemplo 3: promedio y totales generales

`margins=True` agrega una fila y una columna de totales. Aquí calculamos el promedio de cada operación, no la suma.

### Explicación del código: promedio por vendedor y sucursal
- Las filas representan vendedores y las columnas representan sucursales.
- `values="total_venta"` selecciona los importes de venta.
- `aggfunc="mean"` calcula el promedio por operación, no la suma.
- `margins=True` agrega los promedios generales de filas y columnas.
- `margins_name` cambia el nombre de esos totales generales.
- `round(2)` redondea los resultados a dos decimales.

In [ ]:
promedio_vendedor_sucursal = pd.pivot_table(
    df_limpio, values='total_venta', index='vendedor', columns='sucursal',
    aggfunc='mean', fill_value=0, margins=True, margins_name='Promedio general'
).round(2)
promedio_vendedor_sucursal

### Ejemplo 4: varias operaciones en una tabla

También podemos solicitar más de una función de resumen.

### Explicación del código: varias medidas en una tabla
- `index="categoria"` agrupa los resultados por categoría.
- `aggfunc=["sum", "mean", "count"]` solicita tres cálculos: suma de importes, promedio de importes y cantidad de registros.
- `round(2)` limita los decimales.
- La columna `count` cuenta operaciones; no cuenta unidades vendidas.

In [ ]:
tabla_multiple = pd.pivot_table(
    df_limpio, values='total_venta', index='categoria',
    aggfunc=['sum', 'mean', 'count']
).round(2)
tabla_multiple

### `groupby()` frente a `pivot_table()`

| Herramienta | Conviene usarla cuando... |
|---|---|
| `groupby()` | Necesitamos agrupar y producir una serie o tabla lineal |
| `pivot_table()` | Necesitamos comparar dos categorías en filas y columnas |

Ambas pueden obtener resultados equivalentes. La diferencia principal está en la forma de organizar la salida.

## 3. Práctica guiada 1 

Resuelva los siguientes ejercicios. Debajo de cada resultado escriba una oración interpretativa.

1. Cree una tabla con la **facturación total por vendedor y mes**.
2. Cree una tabla con las **unidades vendidas por sucursal y producto**.
3. Calcule el **promedio de venta por categoría y sucursal**.
4. Repita el ejercicio 1 incluyendo totales mediante `margins=True`.
5. Identifique, observando sus tablas, qué vendedor facturó más y qué combinación producto-sucursal vendió más unidades.

### Explicación del código: facturación por vendedor y mes
- `pivot_table()` resume la columna `total_venta`.
- `index="vendedor"` muestra un vendedor por fila.
- `columns="nombre_mes"` muestra los meses en columnas.
- `aggfunc="sum"` suma lo facturado por cada vendedor en cada mes.
- `fill_value=0` reemplaza combinaciones sin ventas por cero.
- La tabla permite comparar el desempeño mensual de los vendedores.

In [14]:
# Ejercicio 1: facturación total por vendedor y mes
ventas_vendedor_mes = df_limpio.pivot_table(
    index="vendedor",
    columns="nombre_mes",
    values="total_venta",
    aggfunc="sum",
    fill_value=0
)

ventas_vendedor_mes


nombre_mes,Abril,Marzo,Mayo
vendedor,,,
Ana,1857000,556000,785000
Carlos,403000,1361000,841000
Lucía,484000,468000,1094000
Miguel,1006000,2005000,536000


### Explicación del código: unidades por sucursal y producto
- Las filas representan sucursales y las columnas representan productos.
- `values="cantidad"` selecciona el número de unidades.
- `aggfunc="sum"` suma las unidades de cada producto en cada sucursal.
- Esta tabla permite detectar dónde se vendió mayor cantidad de cada artículo.

In [15]:
# Ejercicio 2: unidades vendidas por sucursal y producto
unidades_sucursal_producto = df_limpio.pivot_table(
    index="sucursal",
    columns="producto",
    values="cantidad",
    aggfunc="sum",
    fill_value=0
)

unidades_sucursal_producto

producto,Bolígrafo,Calculadora,Carpeta,Cuaderno,Mochila,Regla
sucursal,,,,,,
Centro,43,13,45,34,18,28
Mercado,45,18,50,39,11,14
Terminal,20,22,38,16,11,17


### Explicación del código: promedio y facturación con totales
- La primera parte usa `groupby(["categoria", "sucursal"])` para formar grupos por categoría y sucursal.
- `.mean()` calcula el importe promedio de las operaciones de cada grupo.
- `.unstack()` convierte la sucursal en columnas para facilitar la comparación.
- `fillna(0)`/`fill_value=0` evita celdas vacías y `round(2)` redondea los promedios.
- La segunda parte usa `pivot_table()` para sumar la facturación por vendedor y mes.
- `margins=True` agrega los totales por vendedor, por mes y el total general.

In [17]:
# Ejercicios 3 y 4
promedio_categoria_sucursal = (
    df_limpio.groupby(["categoria", "sucursal"])["total_venta"]
    .mean()
    .unstack(fill_value=0)
    .round(2)
)

promedio_categoria_sucursal



facturacion_con_totales = df_limpio.pivot_table(
    index="vendedor",
    columns="nombre_mes",
    values="total_venta",
    aggfunc="sum",
    fill_value=0,
    margins=True,
    margins_name="Total general"
)

facturacion_con_totales


nombre_mes,Abril,Marzo,Mayo,Total general
vendedor,,,,
Ana,1857000,556000,785000,3198000
Carlos,403000,1361000,841000,2605000
Lucía,484000,468000,1094000,2046000
Miguel,1006000,2005000,536000,3547000
Total general,3750000,4390000,3256000,11396000


### Interpretación de la práctica

- **Hallazgo 1:** Miguel fue el vendedor con mayor facturación total.
- **Hallazgo 2:** La combinación producto-sucursal con más unidades fue Carpeta en Mercado.
- **Evidencia numérica:** Miguel facturó Gs. 3.547.000 y en Mercado se vendieron 50 unidades de Carpetas.
- **Aclaración:** se compara la cantidad de unidades, no el número de operaciones. La tabla de frecuencias responde una pregunta diferente.


### Explicación del código: identificación de máximos
- `groupby("vendedor")["total_venta"].sum()` calcula cuánto facturó cada vendedor.
- `idxmax()` identifica al vendedor con el importe mayor.
- La segunda agrupación utiliza producto y sucursal, y suma `cantidad`.
- `idxmax()` devuelve la combinación producto-sucursal con más unidades.
- Se muestran ambos resultados para respaldar la interpretación de la práctica.

In [18]:
total_por_vendedor = df_limpio.groupby("vendedor")["total_venta"].sum()
mejor_vendedor = total_por_vendedor.idxmax()

unidades_producto_sucursal = (
    df_limpio.groupby(["producto", "sucursal"])["cantidad"]
    .sum()
)

mejor_combinacion = unidades_producto_sucursal.idxmax()

print("Vendedor que más facturó:", mejor_vendedor)
print("Producto y sucursal con más unidades:", mejor_combinacion)

Vendedor que más facturó: Miguel
Producto y sucursal con más unidades: ('Carpeta', 'Mercado')


## 4. Tablas de frecuencia con `pd.crosstab()` 

Una **tabla de frecuencia cruzada** cuenta cuántas veces aparece cada combinación de dos variables categóricas.

Pregunta: **¿Cuántas operaciones registró cada vendedor en cada sucursal?**

### Explicación del código: tabla de frecuencias
- `pd.crosstab()` cuenta registros para cada combinación de vendedor y sucursal.
- El primer argumento define las filas y el segundo las columnas.
- `margins=True` agrega los totales por fila, columna y el total general.
- Esta tabla cuenta operaciones; no suma el importe de las ventas.

In [ ]:
frecuencia_vendedor_sucursal = pd.crosstab(
    df_limpio['vendedor'],
    df_limpio['sucursal'],
    margins=True,
    margins_name='Total'
)
frecuencia_vendedor_sucursal

### Frecuencias porcentuales

Con `normalize='index'` cada fila suma 100 %. Esto ayuda a comparar distribuciones aunque los vendedores tengan distinta cantidad de operaciones.

### Explicación del código: porcentajes por vendedor
- `normalize="index"` convierte cada fila en proporciones cuyo total es 1.
- Multiplicar por 100 transforma esas proporciones en porcentajes.
- `round(1)` redondea a un decimal.
- Cada fila suma aproximadamente 100 %, por lo que se compara cómo distribuye sus operaciones cada vendedor entre las sucursales.

In [ ]:
porcentaje_vendedor_sucursal = pd.crosstab(
    df_limpio['vendedor'], df_limpio['sucursal'], normalize='index'
) * 100
porcentaje_vendedor_sucursal.round(1)

### Diferencia fundamental

- `crosstab()` responde normalmente **cuántos registros existen**.
- `pivot_table()` responde normalmente **cuánto suman, promedian o representan los valores**.

Una persona puede aparecer en muchas operaciones pequeñas y, aun así, no ser quien más facturó.

### Práctica guiada 2

1. Construya una tabla de frecuencias entre `producto` y `sucursal`.
2. Agregue los totales.
3. Obtenga porcentajes por fila.
4. Explique por qué esta tabla no muestra directamente la facturación.

### Explicación del código: frecuencia por producto y sucursal
- `pd.crosstab()` cuenta cuántas operaciones de cada producto se registraron en cada sucursal.
- `index` define los productos como filas y `columns` define las sucursales como columnas.
- No se utiliza `values` porque el objetivo es contar registros, no sumar una columna numérica.

In [19]:
# Resuelva aquí la práctica guiada 2
# 1. Construya una tabla de frecuencias entre `producto` y `sucursal`.
frecuencia_producto = pd.crosstab(
    index=df_limpio["producto"],
    columns=df_limpio["sucursal"]
)

frecuencia_producto

sucursal,Centro,Mercado,Terminal
producto,,,
Bolígrafo,12,11,8
Calculadora,4,6,5
Carpeta,11,9,10
Cuaderno,9,9,5
Mochila,5,2,2
Regla,6,3,3


### Explicación del código: agregar totales
- Se repite la tabla cruzada de producto y sucursal.
- `margins=True` añade una fila y una columna de totales.
- `margins_name="Total"` establece la etiqueta de los totales.
- La esquina inferior derecha muestra el total de operaciones del dataset.

In [21]:
#2. Agregue los totales.
frecuencia_con_total = pd.crosstab(
    df_limpio["producto"],
    df_limpio["sucursal"],
    margins=True,
    margins_name="Total"
)

frecuencia_con_total

sucursal,Centro,Mercado,Terminal,Total
producto,,,,
Bolígrafo,12,11,8,31
Calculadora,4,6,5,15
Carpeta,11,9,10,30
Cuaderno,9,9,5,23
Mochila,5,2,2,9
Regla,6,3,3,12
Total,47,40,33,120


### Explicación del código: porcentajes por producto
- `normalize="index"` calcula la proporción de operaciones de cada producto en cada sucursal, tomando como base el total de ese producto.
- `.mul(100)` convierte las proporciones en porcentajes.
- `.round(1)` redondea a un decimal.
- Por eso, cada fila suma aproximadamente 100 %.

In [24]:
#3. Obtenga porcentajes por fila.
porcentaje_producto = (
    pd.crosstab(
        df_limpio["producto"],
        df_limpio["sucursal"],
        normalize="index"
    )
    .mul(100)
    .round(1)
)

porcentaje_producto

sucursal,Centro,Mercado,Terminal
producto,,,
Bolígrafo,38.70,35.50,25.80
Calculadora,26.70,40.00,33.30
Carpeta,36.70,30.00,33.30
Cuaderno,39.10,39.10,21.70
Mochila,55.60,22.20,22.20
Regla,50.00,25.00,25.00


# 4. Explique por qué esta tabla no muestra directamente la facturación.
La tabla no muestra directamente la facturación porque crosstab() cuenta la cantidad de operaciones, pero no suma el valor de totalventa

## Pausa activa y revisión 

Antes de continuar, complete verbalmente:

1. Una tabla dinámica sirve para resumir y comparar datos organizándolos en filas y columnas.
2. El parámetro aggfunc sirve para indicar qué operación queremos realizar sobre los datos, como suma, promedio o conteo.
3. crosstab() es útil cuando necesitamos contar cuántas veces aparece cada combinación de categorías.
4. Contar operaciones no es igual que sumar facturación porque una operación puede tener un valor diferente a otra, por lo que tener más operaciones no significa necesariamente haber facturado más.

## 5. Visualización de datos 
### Elegir el gráfico correcto

| Pregunta | Gráfico recomendado |
|---|---|
| ¿Qué categoría vende más? | Barras |
| ¿Cómo cambia la venta con el tiempo? | Líneas |
| ¿Qué proporción representa cada categoría? | Circular, con pocas categorías |
| ¿Cómo se comparan categorías entre sucursales? | Barras agrupadas o apiladas |

Un buen gráfico debe tener título claro, nombres de ejes, unidades, escala legible y colores que no confundan.

### Ejemplo 1: gráfico de barras para comparar categorías


### Explicación del código: gráfico de barras horizontales
- `groupby("categoria")["total_venta"].sum()` calcula la facturación de cada categoría.
- `sort_values()` ordena los importes de menor a mayor.
- `.plot(kind="barh")` crea barras horizontales.
- `figsize` establece el tamaño y `color` el color de las barras.
- `set_title()`, `set_xlabel()` y `set_ylabel()` agregan título y nombres de ejes.
- `FuncFormatter` presenta el eje horizontal en guaraníes.
- `tight_layout()` ajusta los espacios y `show()` muestra el gráfico.

In [ ]:
ventas_categoria = df_limpio.groupby('categoria')['total_venta'].sum().sort_values()
ax = ventas_categoria.plot(kind='barh', color='#3478BF', figsize=(9, 5))
ax.set_title('Facturación total por categoría')
ax.set_xlabel('Facturación en guaraníes')
ax.set_ylabel('Categoría')
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'Gs. {x:,.0f}'))
plt.tight_layout()
plt.show()

### Ejemplo 2: gráfico de líneas para observar cambios

Para una secuencia temporal debemos respetar el orden cronológico.

### Explicación del código: gráfico de líneas mensual
- `orden_meses` establece el orden cronológico correcto.
- `groupby("nombre_mes")["total_venta"].sum()` calcula la facturación total de cada mes.
- `reindex()` ordena los resultados como marzo, abril y mayo.
- `kind="line"` crea una línea; `marker="o"` agrega un marcador por mes y `linewidth` define el grosor.
- Se agregan título, ejes y formato monetario para facilitar la lectura.

In [ ]:
orden_meses = ['Marzo', 'Abril', 'Mayo']
ventas_mes = df_limpio.groupby('nombre_mes')['total_venta'].sum().reindex(orden_meses)
ax = ventas_mes.plot(kind='line', marker='o', linewidth=3, color='#2E8B57', figsize=(9, 5))
ax.set_title('Evolución mensual de la facturación')
ax.set_xlabel('Mes')
ax.set_ylabel('Facturación en guaraníes')
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'Gs. {x:,.0f}'))
plt.tight_layout()
plt.show()

### Ejemplo 3: barras agrupadas a partir de una tabla dinámica


### Explicación del código: barras agrupadas
- `tabla_sucursal_categoria` es la tabla dinámica creada anteriormente.
- `.plot(kind="bar")` dibuja una barra por categoría dentro de cada sucursal.
- `legend(title="Categoría")` muestra qué representa cada color.
- La rotación cero mantiene horizontales los nombres de sucursales.
- El formateador muestra los importes en guaraníes.

In [ ]:
ax = tabla_sucursal_categoria.plot(kind='bar', figsize=(11, 6))
ax.set_title('Facturación por sucursal y categoría')
ax.set_xlabel('Sucursal')
ax.set_ylabel('Facturación en guaraníes')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Categoría')
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'Gs. {x:,.0f}'))
plt.tight_layout()
plt.show()

### Ejemplo 4: gráfico circular y uso responsable

El gráfico circular funciona mejor con pocas categorías. Si hay demasiadas porciones, las comparaciones se vuelven difíciles.

### Explicación del código: gráfico circular
- `kind="pie"` crea un gráfico circular con la participación de cada categoría.
- `autopct="%1.1f%%"` escribe el porcentaje con un decimal en cada porción.
- `startangle=90` define el ángulo inicial.
- `ylabel=""` quita la etiqueta vertical que no aporta información.
- Es adecuado aquí porque hay pocas categorías; con muchas porciones sería difícil comparar.

In [ ]:
ventas_categoria.plot(
    kind='pie', autopct='%1.1f%%', startangle=90, figsize=(7, 7),
    ylabel='', title='Participación de cada categoría en la facturación'
)
plt.tight_layout()
plt.show()

### Errores frecuentes que debemos evitar

- usar un gráfico de líneas para categorías sin orden temporal;
- omitir títulos o unidades;
- recargar el gráfico con demasiados colores;
- utilizar un gráfico circular con demasiadas porciones;
- afirmar una causa cuando los datos solo muestran una diferencia;
- modificar la escala para exagerar cambios pequeños.

## 6. Práctica guiada 3: gráficos e interpretación 

Realice las siguientes actividades:

1. Calcule la facturación por producto y represéntela con barras horizontales.
2. Calcule la facturación mensual de cada vendedor y represéntela mediante líneas.
3. Represente las unidades vendidas por sucursal y producto con barras agrupadas.
4. Personalice títulos, nombres de ejes, colores y tamaño.
5. Debajo de cada gráfico escriba un hallazgo que incluya evidencia numérica.

### Explicación del código: facturación por producto

Se agrupan las ventas por producto y se suman los importes. Luego se ordenan y se representan con barras horizontales. `idxmax()` identifica el producto con mayor facturación y `max()` obtiene su importe.

In [ ]:
# Ejercicio gráfico 1: facturación por producto
facturacion_producto = (
    df_limpio.groupby("producto")["total_venta"]
    .sum()
    .sort_values()
)

ax = facturacion_producto.plot(
    kind="barh",
    figsize=(9, 5),
    color="#6b8f71"
)
ax.set_title("Facturación total por producto")
ax.set_xlabel("Facturación en guaraníes")
ax.set_ylabel("Producto")
ax.xaxis.set_major_formatter(
    FuncFormatter(lambda x, pos: f"Gs. {x:,.0f}")
)
plt.tight_layout()
plt.show()

# El producto con mayor facturación y su importe.
producto_mayor_facturacion = facturacion_producto.idxmax()
importe_mayor_producto = facturacion_producto.max()
print(
    f"Mayor facturación: {producto_mayor_facturacion} "
    f"(Gs. {importe_mayor_producto:,.0f})"
)


**Interpretación del gráfico 1:**  
Mochila fue el producto con mayor facturación, con Gs. 3.800.000. En segundo lugar se ubicó Calculadora, con Gs. 3.445.000. Estos valores corresponden a productos individuales, no a categorías.


### Explicación del código: evolución mensual por vendedor

La tabla dinámica coloca los meses en filas y los vendedores en columnas. `aggfunc="sum"` suma los importes de cada mes. `reindex()` fija el orden marzo-abril-mayo. El gráfico de líneas permite observar cómo cambia la facturación de cada vendedor.

In [ ]:
# Ejercicio gráfico 2: evolución mensual por vendedor
orden_meses = ["Marzo", "Abril", "Mayo"]

ventas_mensuales_vendedor = df_limpio.pivot_table(
    index="nombre_mes",
    columns="vendedor",
    values="total_venta",
    aggfunc="sum",
    fill_value=0
).reindex(orden_meses)

ax = ventas_mensuales_vendedor.plot(
    kind="line", marker="o", figsize=(10, 6), linewidth=2
)
ax.set_title("Evolución mensual de la facturación por vendedor")
ax.set_xlabel("Mes")
ax.set_ylabel("Facturación en guaraníes")
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"Gs. {x:,.0f}"))
ax.legend(title="Vendedor")
plt.tight_layout()
plt.show()

print("Facturación mensual por vendedor:")
display(ventas_mensuales_vendedor)


**Interpretación del gráfico 2:**  
El gráfico permite comparar la facturación de cada vendedor entre marzo, abril y mayo. Miguel tuvo su mayor facturación en marzo (Gs. 2.005.000), mientras que Ana alcanzó su valor más alto en abril (Gs. 1.857.000). Las líneas muestran cambios mensuales, pero no explican por sí solas sus causas.


### Explicación del código: unidades por sucursal y producto

La tabla dinámica resume las unidades por sucursal y producto. El gráfico de barras agrupadas permite comparar productos dentro de cada sucursal. Finalmente, `groupby()` y `idxmax()` encuentran la combinación que registra más unidades.

In [ ]:
# Ejercicio gráfico 3: unidades por sucursal y producto
unidades_grafico = df_limpio.pivot_table(
    index="sucursal",
    columns="producto",
    values="cantidad",
    aggfunc="sum",
    fill_value=0
)

ax = unidades_grafico.plot(
    kind="bar", figsize=(11, 6), width=0.8
)
ax.set_title("Unidades vendidas por sucursal y producto")
ax.set_xlabel("Sucursal")
ax.set_ylabel("Unidades vendidas")
ax.legend(title="Producto", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

unidades_por_combinacion = df_limpio.groupby(
    ["producto", "sucursal"]
)["cantidad"].sum()
print("Combinación con más unidades:", unidades_por_combinacion.idxmax())
print("Cantidad de unidades:", unidades_por_combinacion.max())


**Interpretación del gráfico 3:**  
La combinación con mayor cantidad de unidades fue Carpeta en la sucursal Mercado, con 50 unidades. Esta comparación se basa en la suma de la columna `cantidad`, no en el importe facturado.


## 7. Cómo interpretar correctamente 

Una conclusión sólida contiene tres elementos:

1. **Hallazgo:** qué se observa.
2. **Evidencia:** número o comparación que lo demuestra.
3. **Implicación:** por qué puede ser importante.

### Modelo

> La categoría Tecnología presenta la mayor facturación, con Gs. X. Esto indica que, aunque no necesariamente tenga más operaciones, sus ventas aportan una parte importante de los ingresos.

Observe que no afirmamos la causa. Para decir *por qué* ocurrió necesitaríamos otros datos o una investigación adicional.

## 8. Desafío integrador: miniinforme gerencial 

La administración solicita un informe breve para orientar una campaña comercial. Prepare lo siguiente:

1. Una tabla dinámica de facturación por sucursal y categoría.
2. Una tabla de frecuencia de operaciones por vendedor y sucursal.
3. Un gráfico de barras que permita comparar productos.
4. Un gráfico de líneas que muestre la evolución mensual.
5. Tres conclusiones respaldadas por números.
6. Dos recomendaciones concretas para la librería.

### Criterios de logro

| Criterio | Puntaje |
|---|---:|
| Tablas correctas y legibles | 3 |
| Gráficos adecuados y personalizados | 3 |
| Conclusiones con evidencia | 2 |
| Recomendaciones coherentes | 2 |
| **Total** | **10** |

### Explicación del código: desafío integrador

El bloque reúne las respuestas solicitadas: una tabla dinámica de importes, una tabla cruzada que cuenta operaciones, un gráfico de barras por producto y un gráfico de líneas por mes. Al final se imprimen valores máximos y totales que sirven como evidencia para redactar conclusiones y recomendaciones.

In [ ]:
# 1. Tabla dinámica: facturación por sucursal y categoría
tabla_reporte_facturacion = df_limpio.pivot_table(
    index="sucursal", columns="categoria", values="total_venta",
    aggfunc="sum", fill_value=0, margins=True, margins_name="Total"
)
display(tabla_reporte_facturacion)

# 2. Tabla de frecuencia: operaciones por vendedor y sucursal
tabla_reporte_frecuencia = pd.crosstab(
    df_limpio["vendedor"], df_limpio["sucursal"],
    margins=True, margins_name="Total"
)
display(tabla_reporte_frecuencia)

# 3. Gráfico de barras: facturación por producto
reporte_producto = df_limpio.groupby("producto")["total_venta"].sum().sort_values()
ax = reporte_producto.plot(kind="bar", figsize=(10, 5), color="#789b83")
ax.set_title("Facturación por producto")
ax.set_xlabel("Producto")
ax.set_ylabel("Facturación en guaraníes")
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"Gs. {x:,.0f}"))
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

# 4. Gráfico de líneas: evolución mensual de la facturación
reporte_mensual = df_limpio.groupby("nombre_mes")["total_venta"].sum().reindex(
    ["Marzo", "Abril", "Mayo"]
)
ax = reporte_mensual.plot(
    kind="line", marker="o", linewidth=2, figsize=(9, 5), color="#4d7962"
)
ax.set_title("Evolución mensual de la facturación")
ax.set_xlabel("Mes")
ax.set_ylabel("Facturación en guaraníes")
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"Gs. {x:,.0f}"))
plt.tight_layout()
plt.show()

# 5. Valores que sirven como evidencia para el miniinforme
print("Facturación total: Gs.", f"{df_limpio['total_venta'].sum():,.0f}")
print("Producto con mayor facturación:", reporte_producto.idxmax(),
      "- Gs.", f"{reporte_producto.max():,.0f}")
print("Mes con mayor facturación:", reporte_mensual.idxmax(),
      "- Gs.", f"{reporte_mensual.max():,.0f}")
print("Sucursal con mayor facturación:",
      df_limpio.groupby("sucursal")["total_venta"].sum().idxmax())


### Miniinforme

**Conclusión 1:** La sucursal Centro registró la mayor facturación, con Gs. 4.118.000, por encima de Mercado (Gs. 3.840.000) y Terminal (Gs. 3.438.000).

**Conclusión 2:** Marzo fue el mes de mayor facturación, con Gs. 4.390.000. En abril se registraron Gs. 3.750.000 y en mayo Gs. 3.256.000.

**Conclusión 3:** La categoría Accesorios alcanzó la mayor facturación, con Gs. 3.800.000, seguida de Papelería, con Gs. 3.738.000.

**Recomendación 1:** Evaluar una campaña de promoción de productos de Accesorios, porque fue la categoría que generó el mayor importe de ventas en el conjunto analizado.

**Recomendación 2:** Revisar las ventas de abril y mayo para identificar qué productos tuvieron descensos y planificar promociones específicas. Los datos muestran la variación, pero no permiten determinar por sí solos sus causas.


## 9. Cierre y metacognición — 5 minutos

Complete las siguientes frases:

- Hoy aprendí que una tabla dinámica...
- El gráfico más útil para comparar categorías es...
- Antes de formular una conclusión debo...
- Una dificultad que todavía necesito practicar es...

### Lista de verificación

- [ ] Construí una tabla con `pivot_table()`.
- [ ] Utilicé correctamente `aggfunc`.
- [ ] Construí una tabla con `crosstab()`.
- [ ] Elegí el gráfico según la pregunta.
- [ ] Incluí título, ejes y unidades.
- [ ] Redacté conclusiones con evidencia numérica.
- [ ] Evité atribuir causas que los datos no demuestran.